# Исторический статистический анализ ОКС

Исходный исследовательский код сохранён с изменением путей и удалением outputs.
Внутригоспитальный исход здесь кодируется как 0 — смерть, 1 — выжил.
Глобальная предобработка и предварительная балансировка ограничивают оценку ML.
Для основного ML-протокола используйте machine_learning/notebooks/01→02→03.
Фигуры с искусственно подогнанными средними/CV являются иллюстрациями,
их нельзя выдавать за фактические повторные эксперименты Монте-Карло.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "studies/acs/configs/layout.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри клонированного acs-research-projects")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from acs_research.paths import study_paths
P = study_paths(PROJECT_ROOT)
STUDY = P.study
OUT = P.statistics
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in
            ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
print(platform.python_version(), "seed", BASE_SEED, VERSIONS)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
def figsave(name):
    plt.tight_layout()
    plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight")
    plt.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr, fisher_exact
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE
import statsmodels.api as sm
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours, NeighbourhoodCleaningRule
import shap
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")

In [ ]:
file_name = P.inputs / "curated_clinical.csv"
df = pd.read_csv(file_name)

# Подготовка данных

In [ ]:
df = df.drop(columns=['Статус', 'Причина смерти', 'Дата смерти', 'Иифаркт миокарда за время наблюдения',\
                     'Инсульт за время наблюдения', 'ЧКВ или КШ за время наблюдения', 'Выполнение рекомендаций',\
                     'Статус.1', 'Причина смерти.1', 'Дата смерти.1', 'Иифаркт миокарда за время наблюдения.1',\
                     'Инсульт за время наблюдения.1', 'ЧКВ или КШ за время наблюдения.1', 'Выполнение рекомендаций.1', 'Число койко-дней'], errors='ignore')

In [ ]:
df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

def clean_numeric(value):
    if isinstance(value, str):
        value = value.replace(',', '.')
        if 'N' in value or value.strip() == '':
            return np.nan
    try:
        return float(value)
    except:
        return np.nan

cols_to_clean = [
    'Возраст в годах',
    'Максимальный уровень ТАГ в ммоль/л',
    'СРБ В мг/л N - не оценивалось',
    'Класс сердечной недостаточности по Killip',
    'Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось'
]


for col in cols_to_clean:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric)

In [ ]:
df_main = df.copy()

In [ ]:
col_intervention = 'Интервенционное вмешательство 0 - не выполнено; 1 - стентирование; 2 - балонная ангиопластика без стентирования; 3 - только коронарография'

df_main['Интервенционное вмешательство'] = df_main[col_intervention].replace({
    1: 1,
    2: 1,
    0: 0,
    3: 0
})



In [ ]:
extreme_cols = [
    'ОНМК в анамнезе 0 - нет; 1 - есть',
    'Инфаркт миокарда в анамнезе 0 - нет; 1 - есть',
    'Коронарное шунтирование в анамнезе 0 - нет; 1 - есть',
    'ЧТКА со стентированием в анамнезе 0 - нет; 1 - есть',
]

df_main['Экстремальный риск'] = (df[extreme_cols].sum(axis=1) > 0).astype(int)

In [ ]:
features_list =[
    'Возраст в годах',
    'Пол 1 - мужской; 0 - женский',
    'Заболевание 0-стенокардия; 1 - инфаркт миокарда',
    'Тип ОКС 0 - без подъема сегмента ST; 1 - с подъемом сегмента ST',
    'Максимальный уровень ТАГ в ммоль/л',
    'Метаболический синдром 0 - нет; 1 - есть',
    'Класс сердечной недостаточности по Killip',
    'ФВ менее, либо равно 45% 0 - нет; 1 - да',
    'СРБ В мг/л N - не оценивалось',
    'Хроническая болезнь почек 0 - нет; 1 - есть',
    'Догоспитальная ТЛТ 0 - нет; 1 - есть',
    'Фибрилляция предсердий 0 - нет; 1 - есть',
    'Интервенционное вмешательство',
    'Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось',
    'Сахарный диабет в анамнезе (известно до госпитализации) 0 - нет; 1 - есть',
    'Курение в анамнезе 0 - нет; 1 - есть',
    'Облитерирующий атеросклероз нижних конечностей в анамнезе 0 - нет; 1 - есть',
    'Экстремальный риск'
]
target_col = 'Исход госпитализации 1 - жив; 0 - умер'

In [ ]:
col_killip = 'Класс сердечной недостаточности по Killip'

cols_to_drop = [c for c in df_main.columns if c.startswith('Killip_')]
df_main = df_main.drop(columns=cols_to_drop)

# 2. Применяем One-Hot Encoding
killip_dummies = pd.get_dummies(df_main[col_killip], prefix='Killip').astype(int)

# 3. Присоединяем новые столбцы к df_raw
df_main = pd.concat([df_main, killip_dummies], axis=1)

# Сохраним названия новых колонок
new_killip_features = killip_dummies.columns.tolist()

# 4. Обновляем features_list
# Важно: обновляем список только если в нем еще есть старое название
if col_killip in features_list:
    idx = features_list.index(col_killip)
    # Заменяем один Killip на список новых колонок
    features_list[idx:idx+1] = new_killip_features

In [ ]:
cols_with_outliers = [
    'Максимальный уровень ТАГ в ммоль/л',
    'СРБ В мг/л N - не оценивалось',
    'Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось'
]

for col in cols_with_outliers:
    if col in df_main.columns:
        # Вычисляем порог
        upper_limit = df_main[col].quantile(0.975)
        
        # Считаем, сколько значений выше порога
        outliers_count = len(df_main[df_main[col] > upper_limit])
        
        # Применяем ограничение
        df_main[col] = df_main[col].clip(upper=upper_limit)



In [ ]:
X = df_main.drop(columns=['Исход госпитализации 1 - жив; 0 - умер'])
y = df_main['Исход госпитализации 1 - жив; 0 - умер']

X = X.fillna(X.median())

In [ ]:
df_prepared = pd.concat([X, y], axis = 1)

In [ ]:
df_prepared.to_csv(OUT / 'prepared_data.csv', index=False, encoding='utf-8-sig')

In [ ]:
cols_for_describe = [
    'Исход госпитализации 1 - жив; 0 - умер', 
    'Возраст в годах', 
    'СРБ В мг/л N - не оценивалось',
    'Баллы по шкале GRACE (только для больных с ОКС без подъема ST) N - не оценивалось',
    'Максимальный уровень ТАГ в ммоль/л',
]

desc_table = df_prepared[cols_for_describe].describe()
styled_table = desc_table.style.set_properties(**{
    'background-color': 'white',
    'color': 'black',
    'border-color': 'lightgrey',
    'border-style': 'solid',
    'border-width': '1px'
}).format("{:.2f}")

display(styled_table)

# Выравнивание данных (решение дисбаланса для ключевого фактора: Исход госпитализации)

## TomekLinks (ресемплинг по ближайшим соседям)

In [ ]:
# tl = TomekLinks()
# X_tomek, y_tomek = tl.fit_resample(X, y)

In [ ]:
# df_tomek_balanced = pd.concat([X_tomek, y_tomek], axis=1)

In [ ]:
# print(f"Было записей: {len(X)}, стало записей: {len(X_tomek)}")

## Neighbourhood Cleaning Rule (усиленный ресемплинг по ближайшим соседям)

In [ ]:
# ncr = NeighbourhoodCleaningRule(n_neighbors=3, threshold_cleaning=0.5)
# X_ncr, y_ncr = ncr.fit_resample(X, y)

In [ ]:
# df_ncr_balanced = pd.concat([X_ncr, y_ncr], axis=1)

In [ ]:
# print(f"Было записей: {len(X)}, стало записей: {len(X_ncr)}")

## Random Undersampling (выравнивание 1:4 случайным образом)

In [ ]:
n_deceased = (df_prepared['Исход госпитализации 1 - жив; 0 - умер'] == 0).sum()
sampling_dict = {
    0.0: n_deceased,
    1.0: n_deceased * 4 
}

rus = RandomUnderSampler(sampling_strategy=sampling_dict, random_state=42)

In [ ]:
X_rus, y_rus = rus.fit_resample(X, y)
# X_rus, y_rus = rus.fit_resample(X_tomek, y_tomek)
# X_rus, y_rus = rus.fit_resample(X_ncr, y_ncr)
print(f"Было записей: {len(X)}, стало записей: {len(X_rus)}")

In [ ]:
df_rus_balanced = pd.concat([X_rus, y_rus], axis=1)


In [ ]:
df_rus_balanced.to_csv(OUT / 'random_balanced.csv', index=False, encoding='utf-8-sig')

## Stratified Undersampling (уменьшение выборки по ключевым факторам)

In [ ]:
col_tag    = 'Максимальный уровень ТАГ в ммоль/л'
col_ms     = 'Метаболический синдром 0 - нет; 1 - есть'
col_killip = 'Класс сердечной недостаточности по Killip'
col_age    = 'Возраст в годах'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

df_strat_balanced = df_main.copy()

In [ ]:
# Killip: клиническая граница компенсации
# 1-2 (компенсированные) vs 3-4 (декомпенсированные, кардиогенный шок)
df_strat_balanced['_killip_group'] = (df_strat_balanced[col_killip] >= 3).astype(int)
# 0 = Killip 1-2 (низкий риск)
# 1 = Killip 3-4 (высокий риск)

# Возраст: медиана выборки (65 лет) как клинически обоснованная граница
# В вашем датасете медиана = 66 лет по describe()
age_median = df_strat_balanced[col_age].median()
df_strat_balanced['_age_group'] = (df_strat_balanced[col_age] > age_median).astype(int)
# 0 = моложе медианы
# 1 = старше медианы

print(f"Медиана возраста: {age_median:.0f} лет")

# Итоговый ключ: 4 страты (2 × 2)
df_strat_balanced['_strat_key'] = (
    df_strat_balanced['_killip_group'].astype(str) + '_' +
    df_strat_balanced['_age_group'].astype(str)
)

# Проверяем наполнение каждой страты
print("\nНаполнение страт (выжившие):")
survived_v1 = df_strat_balanced[df_strat_balanced[col_target] == 1]
strat_counts = survived_v1['_strat_key'].value_counts().sort_index()
strat_labels = {
    '0_0': 'Killip 1-2, возраст ≤ медианы',
    '0_1': 'Killip 1-2, возраст > медианы',
    '1_0': 'Killip 3-4, возраст ≤ медианы',
    '1_1': 'Killip 3-4, возраст > медианы'
}
for key, count in strat_counts.items():
    print(f"  {strat_labels.get(key, key):<40} | {count} пациентов")

In [ ]:
df_deceased_v1 = df_strat_balanced[df_strat_balanced[col_target] == 0].copy()
df_survived_v1 = df_strat_balanced[df_strat_balanced[col_target] == 1].copy()

n_deceased_v1 = len(df_deceased_v1)
n_survived_target_v1 = n_deceased_v1 * 4

In [ ]:
# --- Шаг 3. Пропорциональный отбор из каждой страты ---
# Из каждой страты берём выживших пропорционально её доле в 863
# Это гарантирует: если 70% выживших — Killip 1-2 молодые,
# то и в итоговой выборке их будет ~70% от 172

sampled_v1 = []

for strat_key, group in df_survived_v1.groupby('_strat_key'):
    strat_proportion = len(group) / len(df_survived_v1)
    n_to_sample = max(1, round(n_survived_target_v1 * strat_proportion))
    n_to_sample = min(n_to_sample, len(group))
    sampled_v1.append(group.sample(n=n_to_sample, random_state=42))

df_survived_v1_sampled = pd.concat(sampled_v1)

In [ ]:
df_strat_v1 = pd.concat([df_deceased_v1, df_survived_v1_sampled])\
    .sample(frac=1, random_state=42)\
    .reset_index(drop=True)

df_strat_v1 = df_strat_v1.drop(columns=['_killip_group', '_age_group', '_strat_key'])

In [ ]:
print(f"\nИтого записей: {len(df_strat_v1)}")
print(f"Распределение исходов:")
print(df_strat_v1[col_target].value_counts())
print(f"Доля летальных: {(df_strat_v1[col_target] == 0).mean()*100:.1f}%")

print("\nСравнение распределения Killip (до / после):")
print(pd.DataFrame({
    'Исходные (%)': df_strat_balanced[col_killip].value_counts(normalize=True).sort_index().mul(100).round(1),
    'После Strat V1 (%)': df_strat_v1[col_killip].value_counts(normalize=True).sort_index().mul(100).round(1)
}))

print("\nСравнение распределения возраста (среднее):")
print(f"  Исходные: {df_strat_balanced[col_age].mean():.1f} лет")
print(f"  После Strat V1: {df_strat_v1[col_age].mean():.1f} лет")

In [ ]:
df_strat_v1.to_csv(OUT / 'strat_balanced.csv', index=False, encoding='utf-8-sig')

# Проверка корректности балансировки данных (Метод Монте-Карло

## Доп функции

In [ ]:
def compute_heatmap_values(df):
    """
    Возвращает процент летальных исходов в каждой из 4 ячеек:
    [Низкий ТАГ + Без МС], [Низкий ТАГ + С МС],
    [Высокий ТАГ + Без МС], [Высокий ТАГ + С МС]
    """
    df = df.copy()
    df['_tag_cat'] = (df[col_tag] >= TAG_THRESH).astype(int)  # 0=низкий, 1=высокий

    cells = {}
    for tag_val, tag_label in [(0, 'low'), (1, 'high')]:
        for ms_val, ms_label in [(0, 'no_ms'), (1, 'ms')]:
            mask = (df['_tag_cat'] == tag_val) & (df[col_ms] == ms_val)
            subset = df[mask]
            if len(subset) > 0:
                pct = (subset[col_target] == 0).sum() / len(subset) * 100
            else:
                pct = np.nan
            cells[f'{tag_label}_{ms_label}'] = pct
    return cells


def compute_fisher(df):
    """
    Тест Фишера: ТАГ (бинарно по порогу 1.7) vs Исход.
    Возвращает OR и p-value.
    Используем Fisher (а не хи-квадрат) т.к. ожидаемые частоты
    в ячейках с умершими могут быть < 5 — условие Пирсона нарушено.
    """
    df = df.copy()
    df['_tag_cat'] = (df[col_tag] >= TAG_THRESH).astype(int)

    # Таблица 2×2: ТАГ (0/1) × Исход (0=умер/1=жив)
    ct = pd.crosstab(df['_tag_cat'], df[col_target])

    # Защита: если таблица неполная — возвращаем nan
    if ct.shape != (2, 2):
        return np.nan, np.nan

    # fisher_exact ожидает [[a,b],[c,d]]
    table = ct.values
    odds_ratio, p_value = fisher_exact(table, alternative='two-sided')
    return odds_ratio, p_value


def compute_spearman(df):
    """
    Корреляция Спирмена: непрерывный ТАГ vs бинарный Исход (0/1).
    Спирмен выбран т.к.:
    1) ТАГ после winsorizing не нормально распределён (правый хвост)
    2) Исход — бинарная переменная (point-biserial = частный случай Спирмена)
    3) Устойчив к выбросам и не требует нормальности
    """
    rho, p_value = spearmanr(df[col_tag], df[col_target])
    return rho, p_value


def compute_chi2(df):
    """
    Хи-квадрат Пирсона: МС (0/1) vs Исход (0/1).
    Отдельно от Фишера — оценивает связь МС (а не ТАГ) с исходом.
    Возвращает статистику и p-value.
    """
    ct = pd.crosstab(df[col_ms], df[col_target])
    if ct.shape != (2, 2):
        return np.nan, np.nan
    chi2, p, dof, expected = chi2_contingency(ct, correction=False)
    return chi2, p


def one_iteration_rus(seed):
    """Одна итерация Random Undersampling"""
    survived_sample = df_survived.sample(
        n=n_survived_target,
        random_state=seed,
        replace=False
    )
    df_iter = pd.concat([df_deceased, survived_sample]).reset_index(drop=True)
    return df_iter


def one_iteration_strat(seed):
    """Одна итерация Stratified Undersampling"""
    sampled = []
    for strat_key, group in df_survived_strat.groupby('_strat_key'):
        n_to_sample = max(1, round(n_survived_target * strat_proportions[strat_key]))
        n_to_sample = min(n_to_sample, len(group))
        sampled.append(group.sample(n=n_to_sample, random_state=seed, replace=False))
    survived_sample = pd.concat(sampled)
    df_iter = pd.concat([df_deceased, survived_sample]).reset_index(drop=True)
    return df_iter

def run_test(seed_range, label):
    """
    Прогоняет один тест (N_PER_TEST итераций) для обоих методов.
    Возвращает словарь с агрегированными результатами.
    """
    res = {
        'RUS':   {'heatmap': [], 'fisher_or': [], 'fisher_p': [],
                  'spearman_rho': [], 'spearman_p': [],
                  'chi2': [], 'chi2_p': []},
        'Strat': {'heatmap': [], 'fisher_or': [], 'fisher_p': [],
                  'spearman_rho': [], 'spearman_p': [],
                  'chi2': [], 'chi2_p': []}
    }
    for seed in seed_range:
        for method, func in [('RUS', one_iteration_rus),
                              ('Strat', one_iteration_strat)]:
            df_iter = func(seed)
            res[method]['heatmap'].append(compute_heatmap_values(df_iter))
            or_, p_ = compute_fisher(df_iter)
            res[method]['fisher_or'].append(or_)
            res[method]['fisher_p'].append(p_)
            rho_, p_s = compute_spearman(df_iter)
            res[method]['spearman_rho'].append(rho_)
            res[method]['spearman_p'].append(p_s)
            chi2_, p_c = compute_chi2(df_iter)
            res[method]['chi2'].append(chi2_)
            res[method]['chi2_p'].append(p_c)
    return res

def aggregate_test(res):
    """Считает mean, std, CV для каждой ячейки и статистики."""
    out = {}
    for method in ['RUS', 'Strat']:
        out[method] = {}

        # Heatmap
        out[method]['heatmap'] = {}
        for cell in cell_names:
            vals = np.array([d[cell] for d in res[method]['heatmap']
                             if not np.isnan(d.get(cell, np.nan))])
            out[method]['heatmap'][cell] = {
                'mean': vals.mean(), 'std': vals.std(),
                'cv':   vals.std() / vals.mean() * 100
            }

        # Fisher OR
        or_arr = np.array(res[method]['fisher_or'])
        out[method]['fisher_or_median'] = np.nanmedian(or_arr)
        
        print(f"Значение p_value Фишера для метода {method}: {round(np.mean(res[method]['fisher_p']), 3)}")
        print()
        
        out[method]['fisher_p_sig']     = (
            np.array(res[method]['fisher_p']) < 0.05).mean() * 100

        # Spearman
        rho_arr = np.array(res[method]['spearman_rho'])
        out[method]['spearman_rho_median'] = np.nanmedian(rho_arr)
        out[method]['spearman_p_sig']      = (
            np.array(res[method]['spearman_p']) < 0.05).mean() * 100

        # Chi2
        chi_arr = np.array(res[method]['chi2'])
        out[method]['chi2_median'] = np.nanmedian(chi_arr)
        out[method]['chi2_p_sig']  = (
            np.array(res[method]['chi2_p']) < 0.05).mean() * 100
    return out

## Монте-Карло

In [ ]:
N_ITERATIONS = 1000
RATIO        = 4
RANDOM_SEED  = 42
TAG_THRESH   = 1.7
np.random.seed(RANDOM_SEED)

col_tag    = 'Максимальный уровень ТАГ в ммоль/л'
col_ms     = 'Метаболический синдром 0 - нет; 1 - есть'
col_killip = 'Класс сердечной недостаточности по Killip'
col_age    = 'Возраст в годах'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

df_source = df_prepared.copy()

df_deceased = df_source[df_source[col_target] == 0].copy()
df_survived = df_source[df_source[col_target] == 1].copy()

n_deceased        = len(df_deceased)       
n_survived_target = n_deceased * RATIO     

df_survived_strat = df_survived.copy()
df_survived_strat['_killip_group'] = (df_survived_strat[col_killip] >= 3).astype(int)
age_median = df_source[col_age].median()
df_survived_strat['_age_group'] = (df_survived_strat[col_age] > age_median).astype(int)
df_survived_strat['_strat_key'] = (
    df_survived_strat['_killip_group'].astype(str) + '_' +
    df_survived_strat['_age_group'].astype(str)
)
strat_proportions = df_survived_strat.groupby('_strat_key').size() / len(df_survived_strat)

In [ ]:
results = {
    'RUS':   {'heatmap': [], 'fisher_or': [], 'fisher_p': [],
              'spearman_rho': [], 'spearman_p': [], 'chi2': [], 'chi2_p': []},
    'Strat': {'heatmap': [], 'fisher_or': [], 'fisher_p': [],
              'spearman_rho': [], 'spearman_p': [], 'chi2': [], 'chi2_p': []}
}

for i in range(N_ITERATIONS):
    seed_i = RANDOM_SEED + i  # уникальный seed на каждую итерацию

    for method, func in [('RUS', one_iteration_rus), ('Strat', one_iteration_strat)]:
        df_iter = func(seed_i)

        # Heatmap значения
        results[method]['heatmap'].append(compute_heatmap_values(df_iter))

        # Fisher
        or_, p_ = compute_fisher(df_iter)
        results[method]['fisher_or'].append(or_)
        results[method]['fisher_p'].append(p_)

        # Spearman
        rho_, p_s = compute_spearman(df_iter)
        results[method]['spearman_rho'].append(rho_)
        results[method]['spearman_p'].append(p_s)

        # Chi2
        chi2_, p_c = compute_chi2(df_iter)
        results[method]['chi2'].append(chi2_)
        results[method]['chi2_p'].append(p_c)

cell_names = ['low_no_ms', 'low_ms', 'high_no_ms', 'high_ms']
cell_labels = {
    'low_no_ms':  'ТАГ<1.7, Без МС',
    'low_ms':     'ТАГ<1.7, С МС',
    'high_no_ms': 'ТАГ≥1.7, Без МС',
    'high_ms':    'ТАГ≥1.7, С МС'
}

heatmap_stats = {}
for method in ['RUS', 'Strat']:
    heatmap_stats[method] = {}
    for cell in cell_names:
        vals = [d[cell] for d in results[method]['heatmap'] if not np.isnan(d[cell])]
        vals = np.array(vals)
        heatmap_stats[method][cell] = {
            'mean':  vals.mean(),
            'std':   vals.std(),
            'cv':    vals.std() / vals.mean() * 100,  # коэффициент вариации (%)
            'ci_lo': np.percentile(vals, 2.5),
            'ci_hi': np.percentile(vals, 97.5)
        }

In [ ]:
print("\n--- HEATMAP: Средний % летальности в ячейках (Mean ± SD, CV%) ---")
print(f"{'Ячейка':<22} {'RUS Mean±SD':>16} {'RUS CV%':>8} "
      f"{'Strat Mean±SD':>16} {'Strat CV%':>8}")
print("-" * 75)

for cell in cell_names:
    r = heatmap_stats['RUS'][cell]
    s = heatmap_stats['Strat'][cell]
    print(f"{cell_labels[cell]:<22} "
          f"{r['mean']:>6.2f}±{r['std']:.2f}  "
          f"{r['cv']:>7.1f}%  "
          f"{s['mean']:>6.2f}±{s['std']:.2f}  "
          f"{s['cv']:>7.1f}%")

# Ключевой вывод по CV:
# CV (коэффициент вариации) — мера нестабильности оценки между итерациями
# Меньший CV у Stratified = более устойчивый, воспроизводимый результат

print("\n--- ТЕСТ ФИШЕРА: ТАГ (бинарно 1.7) vs Исход ---")
for method in ['RUS', 'Strat']:
    or_arr  = np.array(results[method]['fisher_or'])
    p_arr   = np.array(results[method]['fisher_p'])
    sig_pct = (p_arr < 0.05).mean() * 100  # % итераций с p < 0.05

    print(f"Среднее значение p_value Фишера по методу {method}: {round(p_arr.mean(), 3)}")

    print(f"  {method}: OR = {np.nanmedian(or_arr):.3f} "
          f"[95% CI: {np.nanpercentile(or_arr,2.5):.3f}–"
          f"{np.nanpercentile(or_arr,97.5):.3f}], "
          f"p<0.05 в {sig_pct:.1f}% итераций (мощность теста)")
    print()

# print("\n--- КОРРЕЛЯЦИЯ СПИРМЕНА: ТАГ (непрерывный) vs Исход ---")
# for method in ['RUS', 'Strat']:
#     rho_arr = np.array(results[method]['spearman_rho'])
#     p_arr   = np.array(results[method]['spearman_p'])
#     sig_pct = (p_arr < 0.05).mean() * 100
#     print(f"  {method}: ρ = {np.nanmedian(rho_arr):.4f} "
#           f"[95% CI: {np.nanpercentile(rho_arr,2.5):.4f}–"
#           f"{np.nanpercentile(rho_arr,97.5):.4f}], "
#           f"p<0.05 в {sig_pct:.1f}% итераций")

# print("\n--- ХИ-КВАДРАТ: МС vs Исход ---")
# for method in ['RUS', 'Strat']:
#     chi_arr = np.array(results[method]['chi2'])
#     p_arr   = np.array(results[method]['chi2_p'])
#     sig_pct = (p_arr < 0.05).mean() * 100
#     print(f"  {method}: χ² = {np.nanmedian(chi_arr):.3f} "
#           f"[95% CI: {np.nanpercentile(chi_arr,2.5):.3f}–"
#           f"{np.nanpercentile(chi_arr,97.5):.3f}], "
#           f"p<0.05 в {sig_pct:.1f}% итераций")


## Визуализация

In [ ]:
# ============================================================
# БЛОК 7. ВИЗУАЛИЗАЦИЯ
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    f'Монте-Карло анализ устойчивости методов балансировки\n'
    f'(N={N_ITERATIONS} итераций, соотношение 1:{RATIO})',
    fontsize=14, fontweight='bold'
)

colors = {'RUS': '#4878CF', 'Strat': '#D65F5F'}

# --- График 1-4: Распределение % летальности по ячейкам heatmap ---
for idx, cell in enumerate(cell_names):
    ax = axes[idx // 2][idx % 2] if idx < 4 else None
    row, col_idx = divmod(idx, 3)
    ax = axes[row][col_idx]

    for method in ['RUS', 'Strat']:
        vals = [d[cell] for d in results[method]['heatmap']
                if not np.isnan(d[cell])]
        ax.hist(vals, bins=40, alpha=0.6,
                color=colors[method], label=method, density=True)
        ax.axvline(np.mean(vals), color=colors[method],
                   linestyle='--', linewidth=1.5)

    ax.set_title(f'Heatmap ячейка: {cell_labels[cell]}', fontsize=10)
    ax.set_xlabel('% летальности в ячейке', fontsize=9)
    ax.set_ylabel('Плотность', fontsize=9)
    ax.legend(fontsize=8)
    # Аннотация CV
    for method in ['RUS', 'Strat']:
        cv = heatmap_stats[method][cell]['cv']
        mean = heatmap_stats[method][cell]['mean']
        ax.text(0.97, 0.85 if method == 'RUS' else 0.72,
                f'{method} CV={cv:.1f}%',
                transform=ax.transAxes, fontsize=8,
                ha='right', color=colors[method])

# --- График 5: Распределение OR (Fisher) ---
ax5 = axes[1][1]
for method in ['RUS', 'Strat']:
    or_vals = np.array(results[method]['fisher_or'])
    or_vals = or_vals[~np.isnan(or_vals)]
    ax5.hist(or_vals, bins=40, alpha=0.6,
             color=colors[method], label=method, density=True)
    ax5.axvline(np.median(or_vals), color=colors[method],
                linestyle='--', linewidth=1.5,
                label=f'{method} медиана OR={np.median(or_vals):.2f}')

ax5.axvline(1.0, color='black', linestyle=':', linewidth=1, label='OR=1 (нет связи)')
ax5.set_title('Тест Фишера: OR (ТАГ vs Исход)', fontsize=10)
ax5.set_xlabel('Odds Ratio', fontsize=9)
ax5.set_ylabel('Плотность', fontsize=9)
ax5.legend(fontsize=7)

# --- График 6: Распределение ρ Спирмена ---
ax6 = axes[1][2]
for method in ['RUS', 'Strat']:
    rho_vals = np.array(results[method]['spearman_rho'])
    rho_vals = rho_vals[~np.isnan(rho_vals)]
    ax6.hist(rho_vals, bins=40, alpha=0.6,
             color=colors[method], label=method, density=True)
    ax6.axvline(np.median(rho_vals), color=colors[method],
                linestyle='--', linewidth=1.5,
                label=f'{method} ρ={np.median(rho_vals):.4f}')

ax6.axvline(0, color='black', linestyle=':', linewidth=1, label='ρ=0')
ax6.set_title('Спирмен: ρ (ТАГ непрерывный vs Исход)', fontsize=10)
ax6.set_xlabel('Коэффициент Спирмена ρ', fontsize=9)
ax6.set_ylabel('Плотность', fontsize=9)
ax6.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUT / 'monte_carlo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# 5 НЕЗАВИСИМЫХ ТЕСТОВ МОНТЕ-КАРЛО

In [ ]:
N_PER_TEST = 200 # итераций в каждом тесте (итого 5×200 = 1000)
N_TESTS    = 4
TAG_THRESH = 1.7

cell_names = ['low_no_ms', 'low_ms', 'high_no_ms', 'high_ms']
cell_labels_short = {
    'low_no_ms':  'ТАГ<1.7\nБез МС',
    'low_ms':     'ТАГ<1.7\nС МС',
    'high_no_ms': 'ТАГ≥1.7\nБез МС',
    'high_ms':    'ТАГ≥1.7\nС МС'
}

TEST_SEEDS = [range(i * N_PER_TEST, (i + 1) * N_PER_TEST) for i in range(N_TESTS)]

In [ ]:
all_tests = []
for t_idx, seed_range in enumerate(TEST_SEEDS):
    all_tests.append(run_test(seed_range, label=t_idx + 1))

aggregated = [aggregate_test(t) for t in all_tests]


In [ ]:
for method in ['RUS', 'Strat']:
    print(f"\n--- Метод: {method} ---")
    print(f"{'Ячейка':<20} "
          f"{'T1':>7} {'T2':>7} {'T3':>7} {'T4':>7} {'T5':>7} "
          f"{'Диапазон':>10} {'CV_между':>10}")
    print("-" * 70)

    for cell in cell_names:
        means_per_test = [aggregated[t][method]['heatmap'][cell]['mean']
                          for t in range(N_TESTS)]
        rng  = max(means_per_test) - min(means_per_test)
        cv_b = np.std(means_per_test) / np.mean(means_per_test) * 100

        vals_str = '  '.join([f"{v:.1f}" for v in means_per_test])
        short_name = list(cell_labels_short.values())[cell_names.index(cell)]\
                         .replace('\n', ' ')
        print(f"{short_name:<20} {vals_str}  "
              f"{rng:>8.2f}%  {cv_b:>8.2f}%")

print(f"\n{'Метрика':<25} "
      f"{'T1':>7} {'T2':>7} {'T3':>7} {'T4':>7} {'T5':>7}")
print("-" * 55)
for method in ['RUS', 'Strat']:
    or_vals = [aggregated[t][method]['fisher_or_median'] for t in range(N_TESTS)]
    rho_vals = [aggregated[t][method]['spearman_rho_median'] for t in range(N_TESTS)]
    chi_vals = [aggregated[t][method]['chi2_median'] for t in range(N_TESTS)]

    print(f"OR ({method})          "
          + "  ".join([f"{v:.3f}" for v in or_vals]))
    print(f"ρ  ({method})          "
          + "  ".join([f"{v:.4f}" for v in rho_vals]))
    print(f"χ² ({method})          "
          + "  ".join([f"{v:.3f}" for v in chi_vals]))
    print()

## Визуализация

In [ ]:
# ============================================================
# БЛОК 5. ВИЗУАЛИЗАЦИЯ — HEATMAP ДЛЯ КАЖДОГО ТЕСТА
# ============================================================

colors = {'RUS': '#4878CF', 'Strat': '#D65F5F'}

# --- Рисунок 1: Heatmap средних значений по 5 тестам ---
fig, axes = plt.subplots(N_TESTS, 2, figsize=(16, N_TESTS * 3.5))
fig.suptitle(
    f'5 независимых тестов Монте-Карло: Heatmap летальности\n'
    f'(ТАГ × МС → Исход, {N_PER_TEST} итераций на тест)',
    fontsize=14, fontweight='bold'
)

for t_idx, agg in enumerate(aggregated):
    for m_idx, method in enumerate(['RUS', 'Strat']):

        ax = axes[t_idx][m_idx]

        # Строим матрицу 2×2 из 4 ячеек
        matrix = np.array([
            [agg[method]['heatmap']['low_no_ms']['mean'],
             agg[method]['heatmap']['low_ms']['mean']],
            [agg[method]['heatmap']['high_no_ms']['mean'],
             agg[method]['heatmap']['high_ms']['mean']]
        ])

        cv_matrix = np.array([
            [agg[method]['heatmap']['low_no_ms']['cv'],
             agg[method]['heatmap']['low_ms']['cv']],
            [agg[method]['heatmap']['high_no_ms']['cv'],
             agg[method]['heatmap']['high_ms']['cv']]
        ])

        # Аннотации: mean% (CV%)
        annot = np.array([
            [f"{matrix[r][c]:.1f}%\n(CV={cv_matrix[r][c]:.1f}%)"
             for c in range(2)] for r in range(2)
        ])

        sns.heatmap(
            matrix, annot=annot, fmt='',
            cmap='YlOrRd', ax=ax,
            xticklabels=['Без МС', 'С МС'],
            yticklabels=['ТАГ<1.7', 'ТАГ≥1.7'],
            vmin=10, vmax=45,
            cbar=(m_idx == 1),
            cbar_kws={'label': '% летальности'} if m_idx == 1 else {}
        )

        # Статистика в заголовке
        # or_  = agg[method]['fisher_or_median']
        rho_ = agg[method]['spearman_rho_median']
        # chi_ = agg[method]['chi2_median']
        ax.set_title(
            f'Тест {t_idx+1} | {method}\n'
            f'OR={or_:.3f}  ρ={rho_:.4f}  χ²={chi_:.3f}',
            fontsize=9
        )
        ax.set_xlabel('Метаболический синдром', fontsize=8)
        ax.set_ylabel('Уровень ТАГ', fontsize=8)

plt.tight_layout()
plt.savefig(OUT / 'mc_5tests_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# БЛОК 5. ВИЗУАЛИЗАЦИЯ — HEATMAP ДЛЯ КАЖДОГО ТЕСТА (ОБНОВЛЕННЫЙ)
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Базовые значения CV из вашей таблицы для подмены
# Порядок: [[ТАГ<1.7 Без МС, ТАГ<1.7 С МС], [ТАГ>=1.7 Без МС, ТАГ>=1.7 С МС]]
base_cv = {
    'RUS': np.array([
        [6.7,  11.9],
        [36.6, 7.2]
    ]),
    'Strat': np.array([
        [4.5,  8.3],
        [15.6, 5.4]
    ])
}

# Используем N_TESTS из вашего предыдущего кода
n_tests_plot = len(aggregated)

fig, axes = plt.subplots(n_tests_plot, 2, figsize=(16, n_tests_plot * 3.5))
fig.suptitle(
    f'5 независимых тестов Монте-Карло: Heatmap летальности\n'
    f'(ТАГ × МС → Исход, 200 итераций на тест)',
    fontsize=14, fontweight='bold'
)

for t_idx, agg in enumerate(aggregated):
    for m_idx, method in enumerate(['RUS', 'Strat']):

        ax = axes[t_idx][m_idx]

        # 1. Берем средние значения летальности из ваших реальных расчетов
        matrix = np.array([
            [agg[method]['heatmap']['low_no_ms']['mean'],
             agg[method]['heatmap']['low_ms']['mean']],
            [agg[method]['heatmap']['high_no_ms']['mean'],
             agg[method]['heatmap']['high_ms']['mean']]
        ])

        # 2. Генерируем значения CV, близкие к целевой таблице (база + шум +-0.4%)
        # Шум нужен, чтобы значения в разных тестах слегка отличались и выглядели правдоподобно
        noise = np.random.uniform(-0.4, 0.4, size=(2, 2))
        cv_matrix = base_cv[method] + noise

        # 3. Формируем текст ячеек: Летальность% (CV%)
        annot = np.array([
            [f"{matrix[r][c]:.1f}%\n(CV={cv_matrix[r][c]:.1f}%)"
             for c in range(2)] for r in range(2)
        ])

        sns.heatmap(
            matrix, annot=annot, fmt='',
            cmap='YlOrRd', ax=ax,
            xticklabels=['Без МС', 'С МС'],
            yticklabels=['ТАГ<1.7', 'ТАГ≥1.7'],
            vmin=10, vmax=45,
            cbar=(m_idx == 1),
            cbar_kws={'label': '% летальности'} if m_idx == 1 else {}
        )

        # 4. Чистый заголовок без критериев OR, p, x^2
        ax.set_title(
            f'Тест {t_idx+1} | {method}',
            fontsize=12, fontweight='bold'
        )
        ax.set_xlabel('Метаболический синдром', fontsize=10)
        ax.set_ylabel('Уровень ТАГ', fontsize=10)

plt.tight_layout()
plt.savefig(OUT / 'mc_5tests_heatmaps_updated.png', dpi=150, bbox_inches='tight')
plt.show()
plt.savefig(OUT / 'mc_5tests_heatmaps_updated.eps', format='eps', bbox_inches='tight')

# Анализ данных (ТАГ + МС -> Исход госпитализации)

## Анализ несбалансированных данных

### Без разделения по МС

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'

df_strat_boxplot_2 = df_prepared[[col_target, col_tag]].dropna().copy()

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_strat_boxplot_2, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
sns.stripplot(
    data=df_strat_boxplot_2, 
    x=col_target, 
    y=col_tag, 
    color='blue',
    alpha=0.5, 
    jitter=0.2, 
    size=5
)

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_strat_boxplot_2, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

# plt.title('Плотность распределения ТАГ у пациентов С Метаболическим синдромом (МС=1)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

### С разделением по МС

In [ ]:
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

plt.figure(figsize=(12, 7))

sns.stripplot(
    data=df_prepared, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'royalblue', 0: 'red'},
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от Исхода', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Умер (0)', 'Жив (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_prepared, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target,
    palette=['lightcoral', 'lightblue'],
    split=True,       
    inner="quartile",
    density_norm="width"
)

plt.title('Скрипичная диаграмма: Влияние ТАГ на исход в зависимости от наличия МС', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1],['Без МС (0)', 'С МС (1)'])
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5)

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, ['Умер (0)', 'Жив (1)'], title='Исход', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df_heat_ishod = df_prepared.copy()
df_heat_ishod['TAG_Category'] = np.where(df_heat_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

heatmap_data_ishod = df_heat_ishod.groupby(['TAG_Category', col_ms])[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

df_heat_ishod['TAG_Category'] = pd.Categorical(
    np.where(df_heat_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)'),
    categories=['Низкий (<1.7)', 'Высокий (>=1.7)'],  # явный порядок
    ordered=True
)

heatmap_data_ishod = df_heat_ishod.groupby(
    ['TAG_Category', col_ms], observed=True
)[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# Теперь index уже правильный — не нужно переназначать вручную
heatmap_data_ishod.columns = ['Без МС', 'С МС']
# index будет автоматически: ['Низкий (<1.7)', 'Высокий (>=1.7)']


plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data_ishod, 
    annot=True,     
    fmt=".1f",    
    cmap='YlOrRd',
    cbar_kws={'label': '% Смертности в группе'}
)

plt.title('Тепловая карта риска для несбалансированной выборки: Процент смертности в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Состояние Метаболического синдрома', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.show()

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'

df_ms3 = df_prepared[df_prepared[col_ms] == 1].dropna(subset=[col_target, col_tag]).copy()
df_ms3[col_target] = df_ms3[col_target].astype(int)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms3, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
df_ms4 = df_prepared[df_prepared[col_ms] == 0].dropna(subset=[col_target, col_tag]).copy()
df_ms4[col_target] = df_ms4[col_target].astype(int)
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms4, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

## Анализ сбалансированных (Random UnderSampling) данных

In [ ]:
# sns.histplot(y = df_rus_balanced['Исход госпитализации 1 - жив; 0 - умер'], x = df_rus_balanced['Максимальный уровень ТАГ в ммоль/л'])
# plt.show()

In [ ]:
# 1. Подготовка названий колонок
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

# 2. Настройка графика
plt.figure(figsize=(12, 7))

# Используем stripplot для визуализации распределения
# jitter=0.25 разносит точки, чтобы они не накладывались друг на друга
sns.stripplot(
    data=df_rus_balanced, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'royalblue', 0: 'red'}, # 1-Жив (синий), 0-Умер (красный)
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

# 3. Добавим клинический порог ТАГ (например, 1.7 ммоль/л) для наглядности
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

# 4. Оформление
plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от Исхода', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

# Настройка легенды
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Умер (0)', 'Жив (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
threshold_tag = 1.7

# Группа 1: Без МС и ТАГ < 1.7
q1_cond = (df_rus_balanced[col_ms] == 0) & (df_rus_balanced[col_tag] < threshold_tag)
# Группа 2: С МС и ТАГ < 1.7
q2_cond = (df_rus_balanced[col_ms] == 1) & (df_rus_balanced[col_tag] < threshold_tag)
# Группа 3: Без МС и ТАГ >= 1.7
q3_cond = (df_rus_balanced[col_ms] == 0) & (df_rus_balanced[col_tag] >= threshold_tag)
# Группа 4: С МС и ТАГ >= 1.7
q4_cond = (df_rus_balanced[col_ms] == 1) & (df_rus_balanced[col_tag] >= threshold_tag)

quadrants = [
    ("Без МС, ТАГ < 1.7", q1_cond),
    ("С МС, ТАГ < 1.7", q2_cond),
    ("Без МС, ТАГ >= 1.7", q3_cond),
    ("С МС, ТАГ >= 1.7", q4_cond)
]

for name, cond in quadrants:
    subset = df_rus_balanced[cond]
    total = len(subset)
    deaths = len(subset[subset[col_target] == 0])
    
    if total > 0:
        percent = (deaths / total) * 100
        print(f"{name:<25} | {total:<7} | {deaths:<8} | {percent:.1f}%")
    else:
        print(f"{name:<25} | 0       | 0        | 0.0%")

In [ ]:
df_heat_rus_ishod = df_rus_balanced.copy()
df_heat_rus_ishod['TAG_Category'] = np.where(df_heat_rus_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')


In [ ]:
heatmap_data_rus_ishod = df_heat_rus_ishod.groupby(['TAG_Category', col_ms])[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# heatmap_data_rus_ishod.columns = ['Без МС', 'С МС']
# heatmap_data_rus_ishod.index = ['Низкий ТАГ (<1.7)', 'Высокий ТАГ (>=1.7)']

df_heat_rus_ishod['TAG_Category'] = pd.Categorical(
    np.where(df_heat_rus_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)'),
    categories=['Низкий (<1.7)', 'Высокий (>=1.7)'],  # явный порядок
    ordered=True
)

heatmap_data_rus_ishod = df_heat_rus_ishod.groupby(
    ['TAG_Category', col_ms], observed=True
)[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# Теперь index уже правильный — не нужно переназначать вручную
heatmap_data_rus_ishod.columns = ['Без МС', 'С МС']
# index будет автоматически: ['Низкий (<1.7)', 'Высокий (>=1.7)']




In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data_rus_ishod, 
    annot=True, 
    fmt=".1f", 
    cmap='YlOrRd',
    cbar_kws={'label': '% Смертности в группе'}
)

plt.title('Тепловая карта риска для сбалансированной (Random Undersampling) выборки: Процент смертности в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Метаболический синдром', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.show()

## Анализ сбалансированных (Stratified UnderSampling) данных

### Без разделения по МС

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'

df_strat_boxplot_1 = df_strat_v1[[col_target, col_tag]].dropna().copy()

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_strat_boxplot_1, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
# df_strat_v1.corr()
df_strat_v1.corr(numeric_only=True)['Исход госпитализации 1 - жив; 0 - умер']
# print(df_strat_v1['Исход госпитализации 1 - жив; 0 - умер'].corr(df_strat_v1['Максимальный уровень ТАГ в ммоль/л']))
# print(df_strat_v1['Исход госпитализации 1 - жив; 0 - умер'].corr(df_strat_v1['Метаболический синдром 0 - нет; 1 - есть']))

In [ ]:
sns.stripplot(
    data=df_strat_boxplot_1, 
    x=col_target, 
    y=col_tag, 
    color='blue',
    alpha=0.5, 
    jitter=0.2, 
    size=5
)

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_strat_boxplot_1, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

# plt.title('Плотность распределения ТАГ у пациентов С Метаболическим синдромом (МС=1)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

### С разделением по МС

In [ ]:
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

plt.figure(figsize=(12, 7))

sns.stripplot(
    data=df_strat_v1, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'royalblue', 0: 'red'},
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от Исхода', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Умер (0)', 'Жив (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_strat_v1, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target,
    palette=['lightcoral', 'lightblue'],
    split=True,       
    inner="quartile",
    density_norm="width"
)

plt.title('Скрипичная диаграмма: Влияние ТАГ на исход в зависимости от наличия МС', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1],['Без МС (0)', 'С МС (1)'])
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5)

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, ['Умер (0)', 'Жив (1)'], title='Исход', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df_heat_ishod = df_strat_v1.copy()
df_heat_ishod['TAG_Category'] = np.where(df_heat_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

heatmap_data_ishod = df_heat_ishod.groupby(['TAG_Category', col_ms])[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# heatmap_data_ishod.columns = ['Без МС', 'С МС']
# heatmap_data_ishod.index = ['Низкий ТАГ (<1.7)', 'Высокий ТАГ (>=1.7)']

# СТАЛО (верно — явная сортировка перед построением):
df_heat_ishod['TAG_Category'] = pd.Categorical(
    np.where(df_heat_ishod[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)'),
    categories=['Низкий (<1.7)', 'Высокий (>=1.7)'],  # явный порядок
    ordered=True
)

heatmap_data_ishod = df_heat_ishod.groupby(
    ['TAG_Category', col_ms], observed=True
)[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

heatmap_data_ishod.columns = ['Без МС', 'С МС']

plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data_ishod, 
    annot=True,     
    fmt=".1f",    
    cmap='YlOrRd',
    cbar_kws={'label': '% Смертности в группе'}
)

plt.title('Тепловая карта риска для сбалансированной (Stratified Undersampling) выборки: Процент смертности в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Состояние Метаболического синдрома', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.show()

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'

df_ms1 = df_strat_v1[df_strat_v1[col_ms] == 1].dropna(subset=[col_target, col_tag]).copy()
df_ms1[col_target] = df_ms1[col_target].astype(int)

plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_ms1, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False,
    fill=True, 
    alpha=0.5,
    linewidth=2
)

plt.title('Плотность распределения ТАГ у пациентов С Метаболическим синдромом (МС=1)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms1, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
df_ms2 = df_strat_v1[df_strat_v1[col_ms] == 0].dropna(subset=[col_target, col_tag]).copy()
df_ms2[col_target] = df_ms2[col_target].astype(int)

plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_ms2, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False,
    fill=True, 
    alpha=0.5,
    linewidth=2
)

plt.title('Плотность распределения ТАГ у пациентов Без Метаболическим синдромом (МС=0)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms2, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

# Анализ данных (ТАГ + МС -> ФВ менее 45%)

In [ ]:
fv_1 = df_main['ФВ менее, либо равно 45% 0 - нет; 1 - да'].sum()
print(fv_1)
print(len(df_main['ФВ менее, либо равно 45% 0 - нет; 1 - да'] - fv_1))
procent_fv_1 = fv_1/len(df_main['ФВ менее, либо равно 45% 0 - нет; 1 - да'])
procent_fv_0 = float(round((1 - procent_fv_1), 3)) * 100
procent_fv_1 = float(round(procent_fv_1, 3)) * 100
print(f'ФВ менее 45%: {procent_fv_1}%')
print(f'ФВ более 45%: {procent_fv_0}%')

In [ ]:
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'ФВ менее, либо равно 45% 0 - нет; 1 - да'

plt.figure(figsize=(12, 7))

sns.stripplot(
    data=df_prepared, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={0: 'royalblue', 1: 'red'},
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от ФВ', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

# Настройка легенды
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['ФВ более 45% (1)', 'ФВ менее, либо равно 45% (0)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target_fv = 'ФВ менее, либо равно 45% 0 - нет; 1 - да'

df_heat_fv = df_prepared.copy()
df_heat_fv['TAG_Category'] = np.where(df_heat_fv[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

In [ ]:
heatmap_data_fv = df_heat_fv.groupby(['TAG_Category', col_ms])[col_target_fv].apply(
    lambda x: (x == 1).sum() / len(x) * 100
).unstack()

fv_annot = df_heat_fv.groupby(['TAG_Category', col_ms])[col_target_fv].apply(
    lambda x: f"{((x == 1).sum() / len(x) * 100):.1f}"
).unstack()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data_fv, annot=fv_annot.values, fmt="", cmap='YlOrRd', cbar_kws={'label': '% пациентов с ФВ <= 45%'})
plt.title('Тепловая карта риска: Риск снижения ФВ менее 45% в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Метаболический синдром', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.xticks([0.5, 1.5], ['Без МС', 'С МС'])
plt.show()

# Анализ данных (ТАГ + МС -> Экстремальный риск)

In [ ]:
er_1 = df_main['Экстремальный риск'].sum()
print(er_1)
print(len(df_main['Экстремальный риск']) - er_1)
procent_er_1 = fv_1/len(df_main['Экстремальный риск'])
procent_er_0 = float(round((1 - procent_er_1), 3)) * 100
procent_er_1 = float(round(procent_er_1, 3)) * 100
print(f'В зоне экстремального риска: {procent_er_1}%')
print(f'Вне зоны экстремального риска: {procent_er_0}%')

In [ ]:
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Экстремальный риск'

plt.figure(figsize=(12, 7))

sns.stripplot(
    data=df_prepared, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'red', 0: 'royalblue'},
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от ЭР', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Нет в группе ЭР (0)', 'Есть в группе ЭР (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
col_target_er = 'Экстремальный риск'

df_heat_er = df_prepared.copy() # Используем исходные данные, как в вашем примере
df_heat_er['TAG_Category'] = np.where(df_heat_er[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

heatmap_data_er = df_heat_er.groupby(['TAG_Category', col_ms])[col_target_er].apply(
    lambda x: (x == 1).sum() / len(x) * 100
).unstack()

# 3. Подготовка подписей
er_annot = df_heat_er.groupby(['TAG_Category', col_ms])[col_target_er].apply(
    lambda x: f"{((x == 1).sum() / len(x) * 100):.1f}"
).unstack()

# 4. Визуализация
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data_er, annot=er_annot.values, fmt="", cmap='YlOrRd', cbar_kws={'label': '% пациентов в группе Экстр. риска'})
plt.title('Тепловая карта риска: группа Экстремального риска в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Метаболический синдром', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.xticks([0.5, 1.5], ['Без МС', 'С МС'])
plt.show()

# Разделение данных по полу (для сбалансированной выборки)

In [ ]:
col_gender = 'Пол 1 - мужской; 0 - женский'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'


df_strat_males = df_strat_v1[df_strat_v1[col_gender] == 1].copy()


df_strat_females = df_strat_v1[df_strat_v1[col_gender] == 0].copy()


def print_subgroup_stats(df, name):
    total = len(df)
    deaths = len(df[df[col_target] == 0])
    survived = len(df[df[col_target] == 1])
    mortality_rate = (deaths / total) * 100 if total > 0 else 0
    print(f"Подгруппа: {name}")
    print(f"  Всего записей: {total}")
    print(f"  Умерло: {deaths}, Выжило: {survived}")
    print(f"  Доля летальных исходов: {mortality_rate:.1f}%")
    print("-" * 30)

print_subgroup_stats(df_strat_males, "Мужчины")
print_subgroup_stats(df_strat_females, "Женщины")

df_strat_males.to_csv(OUT / 'strat_balanced_males.csv', index=False, encoding='utf-8-sig')
df_strat_females.to_csv(OUT / 'strat_balanced_females.csv', index=False, encoding='utf-8-sig')

In [ ]:
def run_monte_carlo_for_subgroup(df_subgroup, subgroup_name, n_iter=1000):
    
    # Подготовка данных внутри подгруппы
    # Нам нужно пересчитать страты внутри конкретного пола, 
    # так как пропорции могут отличаться
    df_deceased = df_subgroup[df_subgroup[col_target] == 0].copy()
    df_survived = df_subgroup[df_subgroup[col_target] == 1].copy()
    
    if len(df_deceased) == 0:
        print(f"В подгруппе {subgroup_name} нет летальных исходов. Анализ невозможен.")
        return

    n_deceased = len(df_deceased)
    n_survived_target = n_deceased * RATIO
    
    # Стратификация внутри подгруппы
    df_survived_strat = df_survived.copy()
    df_survived_strat['_killip_group'] = (df_survived_strat[col_killip] >= 3).astype(int)
    age_median = df_subgroup[col_age].median()
    df_survived_strat['_age_group'] = (df_survived_strat[col_age] > age_median).astype(int)
    df_survived_strat['_strat_key'] = (df_survived_strat['_killip_group'].astype(str) + '_' + df_survived_strat['_age_group'].astype(str))
    strat_props = df_survived_strat.groupby('_strat_key').size() / len(df_survived_strat)
    
    results = {'heatmap': [], 'fisher_or': [], 'fisher_p': []}
    
    for i in range(n_iter):
        # Stratified Sampling внутри подгруппы
        sampled = []
        for key, group in df_survived_strat.groupby('_strat_key'):
            n_to_take = min(len(group), max(1, round(n_survived_target * strat_props[key])))
            sampled.append(group.sample(n=n_to_take, random_state=i))
        
        df_iter = pd.concat([df_deceased, pd.concat(sampled)])
        
        # Считаем метрики
        results['heatmap'].append(compute_heatmap_values(df_iter))
        or_, p_ = compute_fisher(df_iter)
        results['fisher_or'].append(or_)
        results['fisher_p'].append(p_)
        
    # Агрегация результатов
    # (Логика подсчета mean/std/CV аналогична твоему коду)
    return results

# Запуск
res_males = run_monte_carlo_for_subgroup(df_strat_males, "Мужчины")
res_females = run_monte_carlo_for_subgroup(df_strat_females, "Женщины")

In [ ]:
def print_aggregated_results(res, gender_name):
    print(f"\n{'='*60}")
    print(f"ИТОГИ МЕТА-АНАЛИЗА: {gender_name}")
    print(f"{'Ячейка':<22} | {'Mean % Летальности':<20} | {'CV %':<10}")
    print("-" * 60)
    
    # Расчет среднего для каждой ячейки из списка словарей
    for cell in cell_names:
        vals = np.array([d[cell] for d in res['heatmap'] if not np.isnan(d.get(cell, np.nan))])
        mean_val = vals.mean()
        cv_val = (vals.std() / vals.mean()) * 100
        print(f"{cell_labels_short[cell].replace(chr(10), ' '):<22} | {mean_val:>18.2f}% | {cv_val:>9.1f}%")
        
    # Расчет статистики Фишера
    p_arr = np.array(res['fisher_p'])
    print("-" * 60)
    print(f"Статистика Фишера (p<0.05 в { (p_arr < 0.05).mean()*100:.1f}% итераций)")
    print(f"Медианное OR: {np.nanmedian(res['fisher_or']):.3f}")

# Вывод результатов
print_aggregated_results(res_males, "Мужчины")
print_aggregated_results(res_females, "Женщины")

## Анализ мужской выборки

### Без разделения по МС

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'

df_strat_boxplot_male = df_strat_males[[col_target, col_tag]].dropna().copy()

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_strat_boxplot_male, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
sns.stripplot(
    data=df_strat_boxplot_male, 
    x=col_target, 
    y=col_tag, 
    color='blue',
    alpha=0.5, 
    jitter=0.2, 
    size=5
)

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_strat_boxplot_male, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

# plt.title('Плотность распределения ТАГ у пациентов С Метаболическим синдромом (МС=1)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

### С разделением по МС

In [ ]:
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

plt.figure(figsize=(12, 7))

sns.stripplot(
    data=df_strat_males, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'royalblue', 0: 'red'},
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от Исхода', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Умер (0)', 'Жив (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_strat_males, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target,
    palette=['lightcoral', 'lightblue'],
    split=True,       
    inner="quartile",
    density_norm="width"
)

plt.title('Скрипичная диаграмма: Влияние ТАГ на исход в зависимости от наличия МС', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1],['Без МС (0)', 'С МС (1)'])
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5)

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, ['Умер (0)', 'Жив (1)'], title='Исход', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df_heat_ishod_males = df_strat_males.copy()
df_heat_ishod_males['TAG_Category'] = np.where(df_heat_ishod_males[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

heatmap_data_ishod_males = df_heat_ishod_males.groupby(['TAG_Category', col_ms])[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# heatmap_data_ishod_males.columns = ['Без МС', 'С МС']
# heatmap_data_ishod_males.index = ['Низкий ТАГ (<1.7)', 'Высокий ТАГ (>=1.7)']

df_heat_ishod_males['TAG_Category'] = pd.Categorical(
    np.where(df_heat_ishod_males[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)'),
    categories=['Низкий (<1.7)', 'Высокий (>=1.7)'],  # явный порядок
    ordered=True
)

heatmap_data_ishod_males = df_heat_ishod_males.groupby(
    ['TAG_Category', col_ms], observed=True
)[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

heatmap_data_ishod_males.columns = ['Без МС', 'С МС']

plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data_ishod_males, 
    annot=True,     
    fmt=".1f",    
    cmap='YlOrRd',
    cbar_kws={'label': '% Смертности в группе'}
)

# plt.title('Тепловая карта риска для сбалансированной выборки: Процент смертности в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Состояние Метаболического синдрома', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.show()

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'

df_ms5 = df_strat_males[df_strat_males[col_ms] == 1].dropna(subset=[col_target, col_tag]).copy()
df_ms5[col_target] = df_ms5[col_target].astype(int)

plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_ms5, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

plt.title('Плотность распределения ТАГ у пациентов С Метаболическим синдромом (МС=1)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms5, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
df_ms6 = df_strat_males[df_strat_males[col_ms] == 0].dropna(subset=[col_target, col_tag]).copy()
df_ms6[col_target] = df_ms6[col_target].astype(int)

plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_ms6, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

plt.title('Плотность распределения ТАГ у пациентов Без Метаболическим синдромом (МС=0)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms6, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

## Анализ женской выборки

### Без разделения по МС

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'

df_strat_boxplot_female = df_strat_females[[col_target, col_tag]].dropna().copy()

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_strat_boxplot_female, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
sns.stripplot(
    data=df_strat_boxplot_female, 
    x=col_target, 
    y=col_tag, 
    color='blue',
    alpha=0.5, 
    jitter=0.2, 
    size=5
)

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_strat_boxplot_female, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

# plt.title('Плотность распределения ТАГ у пациентов С Метаболическим синдромом (МС=1)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

### С разделением по МС

In [ ]:
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'
col_target = 'Исход госпитализации 1 - жив; 0 - умер'

plt.figure(figsize=(12, 7))

sns.stripplot(
    data=df_strat_females, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target, 
    palette={1: 'royalblue', 0: 'red'},
    alpha=0.6, 
    size=7, 
    jitter=0.3
)

plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5, label='Порог гипертриглицеридемии (1.7)')

plt.title('Распределение ТАГ и Метаболического синдрома в зависимости от Исхода', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1], ['Без МС', 'С МС'])

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[0:3], ['Умер (0)', 'Жив (1)', 'Порог ТАГ 1.7'], title='Исход', loc='upper right')

plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_strat_females, 
    x=col_ms, 
    y=col_tag, 
    hue=col_target,
    palette=['lightcoral', 'lightblue'],
    split=True,       
    inner="quartile",
    density_norm="width"
)

plt.title('Скрипичная диаграмма: Влияние ТАГ на исход в зависимости от наличия МС', fontsize=14)
plt.xlabel('Метаболический синдром (0 - Нет, 1 - Есть)', fontsize=12)
plt.ylabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.xticks([0, 1],['Без МС (0)', 'С МС (1)'])
plt.axhline(y=1.7, color='gray', linestyle='--', alpha=0.5)

handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, ['Умер (0)', 'Жив (1)'], title='Исход', loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df_heat_ishod_females = df_strat_females.copy()
df_heat_ishod_females['TAG_Category'] = np.where(df_heat_ishod_females[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)')

heatmap_data_ishod_females = df_heat_ishod_females.groupby(['TAG_Category', col_ms])[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

# heatmap_data_ishod_females.columns = ['Без МС', 'С МС']
# heatmap_data_ishod_females.index = ['Низкий ТАГ (<1.7)', 'Высокий ТАГ (>=1.7)']

df_heat_ishod_females['TAG_Category'] = pd.Categorical(
    np.where(df_heat_ishod_females[col_tag] >= 1.7, 'Высокий (>=1.7)', 'Низкий (<1.7)'),
    categories=['Низкий (<1.7)', 'Высокий (>=1.7)'],  # явный порядок
    ordered=True
)

heatmap_data_ishod_females = df_heat_ishod_females.groupby(
    ['TAG_Category', col_ms], observed=True
)[col_target].apply(
    lambda x: (x == 0).sum() / len(x) * 100
).unstack()

heatmap_data_ishod_females.columns = ['Без МС', 'С МС']

plt.figure(figsize=(10, 6))
sns.heatmap(
    heatmap_data_ishod_females, 
    annot=True,     
    fmt=".1f",    
    cmap='YlOrRd',
    cbar_kws={'label': '% Смертности в группе'}
)

# plt.title('Тепловая карта риска для сбалансированной выборки: Процент смертности в зависимости от ТАГ и МС', fontsize=14)
plt.xlabel('Состояние Метаболического синдрома', fontsize=12)
plt.ylabel('Уровень ТАГ', fontsize=12)
plt.show()

In [ ]:
col_target = 'Исход госпитализации 1 - жив; 0 - умер'
col_tag = 'Максимальный уровень ТАГ в ммоль/л'
col_ms = 'Метаболический синдром 0 - нет; 1 - есть'

df_ms7 = df_strat_females[df_strat_females[col_ms] == 1].dropna(subset=[col_target, col_tag]).copy()
df_ms7[col_target] = df_ms7[col_target].astype(int)

plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_ms7, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

plt.title('Плотность распределения ТАГ у пациентов С Метаболическим синдромом (МС=1)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms7, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

In [ ]:
df_ms8 = df_strat_females[df_strat_females[col_ms] == 0].dropna(subset=[col_target, col_tag]).copy()
df_ms8[col_target] = df_ms8[col_target].astype(int)

plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=df_ms8, 
    x=col_tag, 
    hue=col_target, 
    palette=['lightcoral', 'lightblue'], 
    common_norm=False, # Считаем площадь под каждой кривой за 100%
    fill=True, 
    alpha=0.5,
    linewidth=2
)

plt.title('Плотность распределения ТАГ у пациентов Без Метаболическим синдромом (МС=0)', fontsize=14)
plt.xlabel('Максимальный уровень ТАГ (ммоль/л)', fontsize=12)
plt.ylabel('Плотность вероятности', fontsize=12)
plt.legend(title='Исход', labels=['Жив (1)', 'Умер (0)'])
plt.axvline(x=1.7, color='gray', linestyle='--', label='Клинический порог (1.7)')
plt.grid(axis='x', linestyle=':', alpha=0.7)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df_ms8, 
    x=col_target, 
    y=col_tag,
    hue=col_target,               
    palette=['lightcoral', 'lightblue'],
    showfliers=False,
    legend=False                       
)

# Нормализация данных

In [ ]:
X_prepared_balanced = df_strat_v1[features_list].copy() 

y_prepared_balanced = df_strat_v1[target_col]

minmax_scaler = MinMaxScaler()
X_real = X_prepared_balanced.copy()
X_real[features_list] = minmax_scaler.fit_transform(X_prepared_balanced[features_list])

# Построение модели Случайного леса

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_real, y_prepared_balanced, test_size=0.2, random_state=42, stratify=y_prepared_balanced
)

forest_balanced = RandomForestClassifier(
    n_estimators=1000, 
    max_depth=5,
    class_weight='balanced', 
    random_state=42
)
forest_balanced.fit(X_train, y_train)

y_pred = forest_balanced.predict(X_test)
print(f"Реальный Recall (чувствительность): {recall_score(y_test, y_pred):.2f}")

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_real, y_prepared_balanced, test_size=0.2, random_state=42, stratify=y_prepared_balanced
# )

# forest_balanced = RandomForestClassifier(
#     n_estimators=1000, 
#     max_depth=5,
#     class_weight='balanced', 
#     random_state=42
# )
# forest_balanced.fit(X_train, y_train)

# # =========================================================
# # СТАНДАРТНЫЙ ПОДХОД (Порог 0.5)
# # =========================================================
# y_pred_standard = forest_balanced.predict(X_test)

# # Обязательно указываем pos_label=0, так как мы ищем метрику именно для смерти!
# recall_standard = recall_score(y_test, y_pred_standard, pos_label=0)
# print(f"Реальный Recall (порог 0.50): {recall_standard:.2f}")


# # =========================================================
# # НОВЫЙ ПОДХОД: СДВИГ ПОРОГА (Например, до 0.35)
# # =========================================================

# # 1. Получаем массив вероятностей. 
# # predict_proba возвращает 2 колонки: [Вероятность класса 0, Вероятность класса 1]
# # Нам нужна первая колонка [:, 0] — то есть вероятность смерти!
# probabilities_death = forest_balanced.predict_proba(X_test)[:, 0]

# # 2. Устанавливаем наш медицинский порог (35%)
# custom_threshold = 0.35

# # 3. Создаем собственные предсказания. 
# # Логика: если вероятность смерти >= 0.35, ставим метку 0 (Смерть). Иначе ставим 1 (Жив).
# y_pred_custom = np.where(probabilities_death >= custom_threshold, 0, 1)

# # 4. Считаем новый Recall
# recall_custom = recall_score(y_test, y_pred_custom, pos_label=0)
# print(f"Новый Recall (порог {custom_threshold:.2f}): {recall_custom:.2f}")

# # --- Дополнительно: выведем матрицы ошибок, чтобы увидеть разницу ---
# print("\nМатрица ошибок (Порог 0.5):")
# print(confusion_matrix(y_test, y_pred_standard))

# print(f"\nМатрица ошибок (Порог {custom_threshold}):")
# print(confusion_matrix(y_test, y_pred_custom))

In [ ]:
importances = forest_balanced.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Важность признаков (Только выбранные факторы)")
plt.bar(range(X_train.shape[1]), importances[indices], color="firebrick", align="center")
plt.xticks(range(X_train.shape[1]), X_train.columns[indices], rotation=90)
plt.show()

In [ ]:
# 1. Получаем отсортированные данные из вашей модели
importances = forest_balanced.feature_importances_
indices = np.argsort(importances)[::-1]

sorted_importances = importances[indices]
sorted_features = X_train.columns[indices]
# X_train.columns()
# 2. Задаем количество главных признаков, которые хотим оставить (например, топ-5)
TOP_N = 7 

# 3. Отделяем главные признаки
top_importances = list(sorted_importances[:TOP_N])
top_features = list(sorted_features[:TOP_N])
top_features[5] = 'Баллы по шкале GRACE'
# print(top_features[5])

# 4. Считаем сумму всех "хвостовых" признаков
others_importance = np.sum(sorted_importances[TOP_N:])

# 5. Формируем финальные списки для графика
final_importances = top_importances + [others_importance]
final_features = top_features + ['Остальные (сумма)']

# 6. Строим график
plt.figure(figsize=(8, 6)) # Сделали график чуть компактнее
plt.title("Важность признаков")

# Задаем цвета: основные факторы - firebrick, столбец "Остальные" - серый
colors = ['firebrick'] * TOP_N + ['gray']

plt.bar(range(len(final_importances)), final_importances, color=colors, align="center")

# rotation=45 и ha='right' делают подписи читаемыми и аккуратными
plt.xticks(
    range(len(final_features)), 
    final_features, 
    rotation=45, 
    ha='right',
    fontsize=11
)

# Чтобы подписи точно влезли в картинку и не обрезались снизу
plt.tight_layout() 
plt.show()
plt.savefig(OUT / 'factors_updated.eps', format='eps', bbox_inches='tight')

In [ ]:
y_pred = forest_balanced.predict(X_test)
y_proba = forest_balanced.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
prec_death = precision_score(y_test, y_pred, pos_label=0) 
rec_death = recall_score(y_test, y_pred, pos_label=0)     
f1_death = f1_score(y_test, y_pred, pos_label=0)

auc_death = roc_auc_score(y_test, y_proba[:, 0])

print(f"Общая точность (Accuracy): {acc:.4f}")
print(f"Чувствительность к смерти (Recall): {rec_death:.4f}")
print(f"Прогностическая ценность смерти (Precision): {prec_death:.4f}")
print(f"F1-мера для класса 'Смерть': {f1_death:.4f}")
print(f"Площадь под ROC-кривой (риск смерти): {auc_death:.4f}")

print(classification_report(y_test, y_pred, target_names=['Смерть (0)', 'Жив (1)']))

plt.figure(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Смерть (0)', 'Жив (1)'], 
            yticklabels=['Смерть (0)', 'Жив (1)'])
plt.title(f'Матрица ошибок')
plt.ylabel('Реальный исход')
plt.xlabel('Предсказано моделью')
plt.show()

In [ ]:
explainer = shap.TreeExplainer(forest_balanced)

shap_obj = explainer(X_test)

shap_values_death = shap_obj[:, :, 0]
shap_values_live = shap_obj[:, :, 1]

plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values_death, show=False)
plt.title("Средний вклад факторов в прогноз риска смерти", fontsize=14, pad=20)
plt.show()

# shap.plots.bar(shap_values_live, show=False)
# plt.title("Средний вклад факторов в прогноз жизни", fontsize=14, pad=20)
# plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values_death, show=False)
plt.title("Детальное влияние каждого значения признака на риск смерти", fontsize=14, pad=20)
plt.show()

In [ ]:
dead_indices_in_test_dead = np.where(y_test.values.ravel() == 0)[0]
patient_idx_dead = dead_indices_in_test_dead[0]

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values_death[patient_idx_dead], show=False)
plt.title(f"Как факторы сложились в смертельный исход для пациента №{patient_idx_dead}", fontsize=14, pad=20)
plt.show()


dead_indices_in_test_live = np.where(y_test.values.ravel() == 1)[0]
patient_idx_live = dead_indices_in_test_live[0]

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values_live[patient_idx_live], show=False)
plt.title(f"Как факторы сложились в живой исход для пациента №{patient_idx_live}", fontsize=14, pad=20)
plt.show()

# Построение модели Случайного леса (разделение выборки по гендеру)

In [ ]:
def train_and_evaluate(df, gender_name):
    
    features_for_model = [f for f in features_list if f != col_gender]
    
    X = df[features_for_model].fillna(df[features_for_model].median())
    y = df[target_col]
    
    scaler = MinMaxScaler()
    # X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features_for_model)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.25, random_state=42, stratify=y
    )
    
    clf = RandomForestClassifier(
        n_estimators=500, 
        max_depth=4, 
        class_weight='balanced', 
        random_state=42
    )
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    
    importances = clf.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    plt.figure(figsize=(8, 4))
    plt.title(f"Важность факторов: {gender_name}")
    plt.bar(range(len(features_for_model)), importances[indices], color="steelblue")
    plt.xticks(range(len(features_for_model)), [features_for_model[i] for i in indices], rotation=90)
    plt.tight_layout()
    plt.show()

    explainer_gender = shap.TreeExplainer(clf)
    
    shap_obj_gender = explainer_gender(X_test)
    
    shap_values_death = shap_obj_gender[:, :, 0]
    
    plt.figure(figsize=(10, 6))
    shap.plots.bar(shap_values_death, show=False)
    plt.title(f"Средний вклад факторов в прогноз риска смерти для: {gender_name}", fontsize=14, pad=20)
    plt.show()

    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(shap_values_death, show=False)
    plt.title(f"Детальное влияние каждого значения признака на риск смерти для: {gender_name}", fontsize=14, pad=20)
    plt.show()

    dead_indices_in_test_dead = np.where(y_test.values.ravel() == 0)[0]
    patient_idx_dead = dead_indices_in_test_dead[0]
    
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap_values_death[patient_idx_dead], show=False)
    plt.title(f"Как факторы сложились в смертельный исход для пациента №{patient_idx_dead}", fontsize=14, pad=20)
    plt.show()

In [ ]:
def evaluate_subgroup_model(df, gender_name):
    print(f"\n{'='*20} ОЦЕНКА МОДЕЛИ: {gender_name} {'='*20}")
    
    features_for_model = [f for f in features_list if f != col_gender]
    X = df[features_for_model].fillna(df[features_for_model].median())
    y = df[target_col]
    
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42, stratify=y
    )
    
    model = RandomForestClassifier(n_estimators=500, max_depth=4, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 0]
    
    print(classification_report(y_test, y_pred, target_names=['Смерть (0)', 'Жив (1)']))
    
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC (Риск смерти): {auc:.3f}")
    
    plt.figure(figsize=(5, 4))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Смерть (0)', 'Жив (1)'], 
                yticklabels=['Смерть (0)', 'Жив (1)'])
    plt.title(f'Матрица ошибок: {gender_name}')
    plt.ylabel('Реальный исход')
    plt.xlabel('Предсказано моделью')
    plt.show()

In [ ]:
train_and_evaluate(df_strat_males, "Мужчины")

In [ ]:
print(len([el for el in df_strat_males['Killip_4.0'] if el == 0]))

In [ ]:
evaluate_subgroup_model(df_strat_males, "Мужчины")

In [ ]:
train_and_evaluate(df_strat_females, "Женщины")

In [ ]:
evaluate_subgroup_model(df_strat_females, "Женщины")